# 03. Model Dự đoán Thời gian (XGBoost)

**Mục tiêu:** Train XGBoost model để dự đoán thời gian đến động đất tiếp theo

**Input:** train_enhanced.csv
**Output:** Trained model, evaluation metrics

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*60)
print(" XGBOOST - TIME PREDICTION MODEL ")
print("="*60)

## 1. Load Dữ liệu

In [ ]:
# Load data
data_dir = Path('/home/haind/Desktop/earthquake-sequence-mining/haind/data')

train = pd.read_csv(data_dir / 'train_enhanced.csv')
val = pd.read_csv(data_dir / 'val_enhanced.csv')
test = pd.read_csv(data_dir / 'test_enhanced.csv')

# Load config
with open(data_dir / 'features.json', 'r') as f:
    config = json.load(f)

FEATURES = config['NUMERICAL_FEATURES']
TARGET_TIME = config['TARGET_TIME']

print(f"Train: {len(train):,} events")
print(f"Val: {len(val):,} events")
print(f"Test: {len(test):,} events")
print(f"\nFeatures: {len(FEATURES)}")

## 2. Chuẩn bị Datasets

In [ ]:
# Chuẩn bị features và target
X_train = train[FEATURES]
y_train = train[TARGET_TIME]

X_val = val[FEATURES]
y_val = val[TARGET_TIME]

X_test = test[FEATURES]
y_test = test[TARGET_TIME]

# Fill remaining NaN (nếu có)
X_train = X_train.fillna(0)
X_val = X_val.fillna(0)
X_test = X_test.fillna(0)

y_train = y_train.fillna(y_train.median())
y_val = y_val.fillna(y_val.median())
y_test = y_test.fillna(y_test.median())

print(f"\nX_train shape: {X_train.shape}")
print(f"y_train range: {y_train.min():.2f} đến {y_train.max():.2f} ngày")
print(f"\nTarget statistics (train):")
print(y_train.describe())

## 3. Tạo DMatrix cho XGBoost

In [ ]:
# Tạo DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

print("✓ Đã tạo DMatrix!")

## 4. Thiết lập Parameters

In [ ]:
# XGBoost parameters cho time prediction
params = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'random_state': 42,
    'tree_method': 'hist',
    'n_jobs': -1
}

print("Parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")

## 5. Training Model

In [ ]:
# Training
evals_result = {}

print("\n" + "="*60)
print(" TRAINING TIME MODEL ")
print("="*60)

model = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=100,
    evals_result=evals_result,
    verbose_eval=100
)

print(f"\n✓ Training hoàn thành!")
print(f"Best iteration: {model.best_iteration}")
print(f"Best score: {model.best_score:.4f}")

## 6. Training Curve

In [ ]:
# Vẽ training curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# RMSE
axes[0].plot(evals_result['train']['rmse'], label='Train')
axes[0].plot(evals_result['val']['rmse'], label='Validation')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('RMSE (ngày)')
axes[0].set_title('Training Curve - RMSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(evals_result['train']['mae'], label='Train')
axes[1].plot(evals_result['val']['mae'], label='Validation')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('MAE (ngày)')
axes[1].set_title('Training Curve - MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation trên Validation Set

In [ ]:
# Predict trên validation
y_pred_val = model.predict(dval)

# Metrics
r2_val = r2_score(y_val, y_pred_val)
mae_val = mean_absolute_error(y_val, y_pred_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))

print("="*60)
print(" VALIDATION RESULTS ")
print("="*60)
print(f"R²:   {r2_val:.4f}")
print(f"MAE:  {mae_val:.2f} ngày")
print(f"RMSE: {rmse_val:.2f} ngày")

## 8. Evaluation trên Test Set

In [ ]:
# Predict trên test
y_pred_test = model.predict(dtest)

# Metrics
r2_test = r2_score(y_test, y_pred_test)
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

print("="*60)
print(" TEST RESULTS ")
print("="*60)
print(f"R²:   {r2_test:.4f}")
print(f"MAE:  {mae_test:.2f} ngày")
print(f"RMSE: {rmse_test:.2f} ngày")

## 9. Feature Importance

In [ ]:
# Feature importance
importance = model.get_score(importance_type='weight')
importance_df = pd.DataFrame({
    'feature': importance.keys(),
    'importance': importance.values()
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=importance_df.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importance - Time Prediction')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# XGBoost plot importance
xgb.plot_importance(model, max_num_features=15, height=0.8)
plt.title('Feature Importance (XGBoost)')
plt.tight_layout()
plt.show()

## 10. Actual vs Predicted Plot

In [ ]:
# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Validation
axes[0].scatter(y_val, y_pred_val, alpha=0.3, s=10)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
axes[0].set_xlabel('Thời gian thực (ngày)')
axes[0].set_ylabel('Thời gian dự đoán (ngày)')
axes[0].set_title(f'Validation: R² = {r2_val:.4f}, MAE = {mae_val:.2f}')
axes[0].grid(True, alpha=0.3)

# Test
axes[1].scatter(y_test, y_pred_test, alpha=0.3, s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Thời gian thực (ngày)')
axes[1].set_ylabel('Thời gian dự đoán (ngày)')
axes[1].set_title(f'Test: R² = {r2_test:.4f}, MAE = {mae_test:.2f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Residual Analysis

In [ ]:
# Residuals
residuals = y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual plot
axes[0].scatter(y_pred_test, residuals, alpha=0.3, s=10)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted (ngày)')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residual Plot')
axes[0].grid(True, alpha=0.3)

# Histogram
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_xlabel('Residuals')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residuals Distribution (mean={residuals.mean():.2f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 12. Lưu Model

In [ ]:
# Tạo thư mục model
model_dir = Path('/home/haind/Desktop/earthquake-sequence-mining/haind/models')
model_dir.mkdir(exist_ok=True)

# Lưu model
joblib.dump(model, model_dir / 'xgboost_time_model.pkl')

# Lưu results
results = {
    'model': 'xgboost_time',
    'best_iteration': int(model.best_iteration),
    'best_score': float(model.best_score),
    'test_r2': float(r2_test),
    'test_mae': float(mae_test),
    'test_rmse': float(rmse_test),
    'val_r2': float(r2_val),
    'val_mae': float(mae_val),
    'val_rmse': float(rmse_val),
    'features': FEATURES
}

with open(model_dir / 'time_model_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("="*60)
print(" ĐÃ LƯU MODEL! ")
print("="*60)
print(f"Model: {model_dir / 'xgboost_time_model.pkl'}")
print(f"Results: {model_dir / 'time_model_results.json'}")

## 13. Summary

In [ ]:
print("="*60)
print(" SUMMARY - TIME MODEL ")
print("="*60)
print(f"\nValidation Results:")
print(f"  R²:   {r2_val:.4f}")
print(f"  MAE:  {mae_val:.2f} ngày")
print(f"  RMSE: {rmse_val:.2f} ngày")
print(f"\nTest Results:")
print(f"  R²:   {r2_test:.4f}")
print(f"  MAE:  {mae_test:.2f} ngày")
print(f"  RMSE: {rmse_test:.2f} ngày")
print(f"\nTop 5 Features:")
for i, row in importance_df.head(5).iterrows():
    print(f"  - {row['feature']}: {row['importance']:.0f}")
print(f"\nNext step: Run 04_magnitude_model.ipynb")